# Aufgaben 6 – Modell 3: Entscheidungsbäume
## Klassifikationsbaum & Regressionsbaum
**Kurs:** NKDS – Machine Learning | **Dozentin:** Prof. Dr. Jennifer Schoch, DHBW Karlsruhe  
**Datensatz:** LLM - Detect AI Generated Text | **Gruppe:** [Namen eintragen] | **Datum:** [Datum]

---

## Theoretischer Hintergrund

**Entscheidungsbäume** treffen Vorhersagen durch eine Folge von Ja/Nein-Entscheidungen (Splits):
- Jeder **Knoten** stellt eine Bedingung auf ein Feature dar.
- Jedes **Blatt** enthält die finale Vorhersage.
- Der Baum wird gelernt, indem bei jedem Schritt der Split gewählt wird, der die Daten am besten trennt.

**Klassifikationsbaum** → diskrete Zielvariable (z.B. Mensch vs. KI)  
- Splitting-Kriterium: **Gini-Impurity** oder **Entropie (Information Gain)**

**Regressionsbaum** → kontinuierliche Zielvariable (z.B. Textlänge vorhersagen)  
- Splitting-Kriterium: **Mean Squared Error (MSE)**

**Wichtigste Hyperparameter:**
- `max_depth`: Maximale Tiefe des Baums (begrenzt Overfitting)
- `min_samples_split`: Minimale Anzahl Samples, um einen Knoten zu splitten
- `min_samples_leaf`: Minimale Anzahl Samples in einem Blatt


## 0. Setup & Daten laden

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             mean_squared_error, mean_absolute_error, r2_score)

df = pd.read_csv("train_essays.csv").dropna(subset=["text"]).drop_duplicates(subset=["text"])

# Abgeleitete numerische Features für Regression
df["text_length"]     = df["text"].str.len()
df["word_count"]      = df["text"].str.split().str.len()
df["sentence_count"]  = df["text"].str.split(r"[.!?]+").str.len()
df["avg_word_length"] = df["text"].apply(
    lambda t: sum(len(w) for w in t.split()) / max(len(t.split()), 1)
)
df["unique_word_ratio"] = df["text"].apply(
    lambda t: len(set(t.lower().split())) / max(len(t.split()), 1)
)

print(f"Datensatz: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
print(f"Klassenverteilung:\n{df['generated'].value_counts()}")


## Teil 1: Klassifikationsbaum

---

### 1a. Train / Validation / Test Split (Klassifikation)

In [ ]:
X_text = df["text"]
y_cls  = df["generated"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_text, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print(f"Trainingsdaten:    {len(X_train):>5} Samples")
print(f"Validierungsdaten: {len(X_val):>5} Samples")
print(f"Testdaten:         {len(X_test):>5} Samples")


### 1b. Feature Extraction – TF-IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)
print(f"TF-IDF Feature-Matrix (Train): {X_train_tfidf.shape}")


### 1b–1c. Hyperparameter `max_depth` vergleichen

In [ ]:
depths  = [1, 2, 3, 5, 7, 10, 15, None]
results = []

for depth in depths:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42, criterion="gini")
    clf.fit(X_train_tfidf, y_train)

    train_acc = accuracy_score(y_train, clf.predict(X_train_tfidf))
    val_acc   = accuracy_score(y_val,   clf.predict(X_val_tfidf))

    label = str(depth) if depth is not None else "unbegrenzt"
    results.append({"max_depth": label, "Train Acc": round(train_acc, 4), "Val Acc": round(val_acc, 4)})
    print(f"max_depth={label:>12} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")

cls_df = pd.DataFrame(results)
cls_df


### 1f. Visualisierung – Einfluss von max_depth

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(cls_df))
ax.plot(x, cls_df["Train Acc"], marker="o", color="steelblue", label="Train Accuracy")
ax.plot(x, cls_df["Val Acc"],   marker="s", color="tomato",    label="Val Accuracy", linestyle="--")

best_idx = cls_df["Val Acc"].idxmax()
ax.axvline(x=best_idx, color="gray", linestyle=":", label=f"Bestes max_depth = {cls_df.loc[best_idx,'max_depth']}")

ax.set_xticks(list(x))
ax.set_xticklabels(cls_df["max_depth"], rotation=15)
ax.set_xlabel("max_depth")
ax.set_ylabel("Accuracy")
ax.set_title("Klassifikationsbaum: Einfluss von max_depth auf Train vs. Val Accuracy")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Beobachtung:**  
- Sehr kleine Tiefe (z.B. 1–2) → Underfitting: zu einfaches Modell, lernt die Daten nicht ausreichend.  
- Sehr große Tiefe / unbegrenzt → Overfitting: hohe Train-Accuracy, aber schlechtere Val-Accuracy.  
- Optimale Tiefe liegt dazwischen. [Eigenen besten Wert eintragen]


### Baum-Visualisierung (vereinfacht mit max_depth=3)

In [ ]:
# Trainierten Baum mit max_depth=3 visualisieren (gut lesbar)
clf_vis = DecisionTreeClassifier(max_depth=3, random_state=42, criterion="gini")
clf_vis.fit(X_train_tfidf, y_train)

# Wichtigste Features
feature_names = tfidf.get_feature_names_out()
importances   = clf_vis.feature_importances_
top_idx       = importances.argsort()[::-1][:10]
print("Top-10 wichtigste Features:")
for i in top_idx:
    if importances[i] > 0:
        print(f"  '{feature_names[i]}': {importances[i]:.4f}")


In [ ]:
# Baumstruktur als Text ausgeben
print(export_text(clf_vis, feature_names=list(feature_names), max_depth=3))


In [ ]:
# Grafische Darstellung
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(clf_vis, max_depth=3, feature_names=feature_names,
          class_names=["Mensch", "KI"], filled=True, rounded=True,
          fontsize=9, ax=ax)
plt.title("Klassifikationsbaum (max_depth=3, zur Veranschaulichung)")
plt.tight_layout()
plt.show()


### 1d. Bestes Klassifikationsmodell auf Testdaten

In [ ]:
best_depth = cls_df.loc[cls_df["Val Acc"].idxmax(), "max_depth"]
best_depth_val = None if best_depth == "unbegrenzt" else int(best_depth)
print(f"Gewähltes max_depth: {best_depth}")

clf_best = DecisionTreeClassifier(max_depth=best_depth_val, random_state=42, criterion="gini")
clf_best.fit(X_train_tfidf, y_train)

y_pred_test = clf_best.predict(X_test_tfidf)
test_acc    = accuracy_score(y_test, y_pred_test)

print(f"Test Accuracy: {test_acc:.4f}")
print("\nKlassifikationsbericht:")
print(classification_report(y_test, y_pred_test, target_names=["Mensch (0)", "KI (1)"]))


In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Mensch (0)", "KI (1)"]).plot(cmap="Blues")
plt.title(f"Konfusionsmatrix – Klassifikationsbaum (max_depth={best_depth})")
plt.tight_layout()
plt.show()


### 1e. Stichprobe (Klassifikationsbaum)

In [ ]:
sample_idx   = X_test.sample(10, random_state=7).index
sample_texts = X_test.loc[sample_idx]
sample_true  = y_test.loc[sample_idx]
sample_pred  = clf_best.predict(tfidf.transform(sample_texts))

pd.DataFrame({
    "Text (Ausschnitt)": [t[:80] + "..." for t in sample_texts],
    "Wahrer Wert":       sample_true.values,
    "Vorhersage":        sample_pred,
    "Korrekt?":          ["✅" if t==p else "❌" for t, p in zip(sample_true.values, sample_pred)]
})


---
## Teil 2: Regressionsbaum

> **Fragestellung für Regression:**  
> Kann anhand von messbaren Text-Features (Wortanzahl, Satzanzahl, etc.) die **Textlänge** (Anzahl Zeichen) vorhergesagt werden?  
> Dies dient als Übung zum Regressionsbaum – die Zielvariable `text_length` ist kontinuierlich.

---


### 2a. Train / Validation / Test Split (Regression)

In [ ]:
# Numerische Features als Input, Textlänge als Ziel
feature_cols = ["word_count", "sentence_count", "avg_word_length", "unique_word_ratio", "generated"]
X_reg = df[feature_cols]
y_reg = df["text_length"]

X_trainval_r, X_test_r, y_trainval_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
X_train_r, X_val_r, y_train_r, y_val_r = train_test_split(
    X_trainval_r, y_trainval_r, test_size=0.25, random_state=42
)

print(f"Trainingsdaten:    {len(X_train_r):>5} Samples")
print(f"Validierungsdaten: {len(X_val_r):>5} Samples")
print(f"Testdaten:         {len(X_test_r):>5} Samples")

# Kurzübersicht der Zielvariable
print(f"\nTextlänge (Zeichen):")
print(f"  Min:    {y_reg.min():.0f}")
print(f"  Max:    {y_reg.max():.0f}")
print(f"  Mittel: {y_reg.mean():.0f}")


### 2b–2c. max_depth vergleichen (Regression)

In [ ]:
depths_r  = [1, 2, 3, 5, 7, 10, None]
results_r = []

for depth in depths_r:
    reg = DecisionTreeRegressor(max_depth=depth, random_state=42)
    reg.fit(X_train_r, y_train_r)

    train_rmse = mean_squared_error(y_train_r, reg.predict(X_train_r), squared=False)
    val_rmse   = mean_squared_error(y_val_r,   reg.predict(X_val_r),   squared=False)
    val_r2     = r2_score(y_val_r,             reg.predict(X_val_r))

    label = str(depth) if depth is not None else "unbegrenzt"
    results_r.append({"max_depth": label, "Train RMSE": round(train_rmse, 1),
                       "Val RMSE": round(val_rmse, 1), "Val R²": round(val_r2, 4)})
    print(f"max_depth={label:>12} | Train RMSE: {train_rmse:.1f} | Val RMSE: {val_rmse:.1f} | Val R²: {val_r2:.4f}")

reg_df = pd.DataFrame(results_r)
reg_df


**Metriken für Regression:**
- **RMSE** (Root Mean Squared Error): Durchschnittlicher Fehler in Zeichen – kleiner ist besser.
- **R²** (Bestimmtheitsmaß): Anteil der erklärten Varianz – 1.0 = perfekt, 0.0 = nutzlos.


### 2f. Visualisierung – RMSE & R² nach Tiefe

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

x = range(len(reg_df))
ax1.plot(x, reg_df["Train RMSE"], marker="o", color="steelblue", label="Train RMSE")
ax1.plot(x, reg_df["Val RMSE"],   marker="s", color="tomato",    label="Val RMSE", linestyle="--")
ax1.set_xticks(list(x))
ax1.set_xticklabels(reg_df["max_depth"], rotation=15)
ax1.set_xlabel("max_depth")
ax1.set_ylabel("RMSE (Zeichen)")
ax1.set_title("Regressionsbaum: RMSE je Tiefe")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(x, reg_df["Val R²"], marker="D", color="mediumseagreen")
ax2.set_xticks(list(x))
ax2.set_xticklabels(reg_df["max_depth"], rotation=15)
ax2.set_xlabel("max_depth")
ax2.set_ylabel("R² (Validierung)")
ax2.set_title("Regressionsbaum: R² je Tiefe")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### 2d. Bestes Regressionsmodell auf Testdaten

In [ ]:
best_depth_r_label = reg_df.loc[reg_df["Val RMSE"].idxmin(), "max_depth"]
best_depth_r_val   = None if best_depth_r_label == "unbegrenzt" else int(best_depth_r_label)
print(f"Gewähltes max_depth: {best_depth_r_label}")

reg_best = DecisionTreeRegressor(max_depth=best_depth_r_val, random_state=42)
reg_best.fit(X_train_r, y_train_r)

y_pred_reg  = reg_best.predict(X_test_r)
test_rmse   = mean_squared_error(y_test_r, y_pred_reg, squared=False)
test_mae    = mean_absolute_error(y_test_r, y_pred_reg)
test_r2     = r2_score(y_test_r, y_pred_reg)

print(f"\nTest RMSE: {test_rmse:.1f} Zeichen")
print(f"Test MAE:  {test_mae:.1f} Zeichen")
print(f"Test R²:   {test_r2:.4f}")


In [ ]:
# Vorhersage vs. echter Wert
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test_r, y_pred_reg, alpha=0.3, s=10, color="steelblue")
lim = max(y_test_r.max(), y_pred_reg.max())
ax.plot([0, lim], [0, lim], "r--", label="Perfekte Vorhersage")
ax.set_xlabel("Echter Wert (Textlänge in Zeichen)")
ax.set_ylabel("Vorhergesagter Wert")
ax.set_title(f"Regressionsbaum: Vorhersage vs. echter Wert (R²={test_r2:.3f})")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 2e. Stichprobe (Regressionsbaum)

In [ ]:
sample_r = X_test_r.sample(10, random_state=7)
pred_r   = reg_best.predict(sample_r)
true_r   = y_test_r.loc[sample_r.index]

pd.DataFrame({
    "Wort­anzahl":       sample_r["word_count"].values,
    "Echter Wert (Zeichen)":     true_r.values,
    "Vorhersage (Zeichen)":      pred_r.round(0).astype(int),
    "Abweichung":       (pred_r - true_r.values).round(0).astype(int)
})


## 1g. Gesamtfazit & kritische Bewertung

### Klassifikationsbaum

| Bestes max_depth | Val Accuracy | Test Accuracy |
|-----------------|-------------|---------------|
| [eintragen]     | [eintragen] | [eintragen]   |

### Regressionsbaum

| Bestes max_depth | Val RMSE | Test RMSE | Test R² |
|-----------------|----------|-----------|---------|
| [eintragen]     | [eintragen] | [eintragen] | [eintragen] |

### Stärken von Entscheidungsbäumen
- Sehr gut interpretierbar (visuell darstellbar)
- Kein Feature-Scaling notwendig
- Kann sowohl Klassifikation als auch Regression

### Schwächen von Entscheidungsbäumen
- Neigen stark zu Overfitting (tiefer Baum = sehr hohe Train Accuracy, aber schlechte Generalisierung)
- Instabil: Kleine Datenänderungen können den Baum stark verändern
- Für Textdaten mit TF-IDF: Random Forest/Gradient Boosting meist besser

### Unterschied Klassifikations- vs. Regressionsbaum
| Aspekt | Klassifikationsbaum | Regressionsbaum |
|--------|---------------------|-----------------|
| Zielvariable | Diskret (Klassen) | Kontinuierlich |
| Splitting-Kriterium | Gini / Entropie | MSE |
| Blatt-Vorhersage | Mehrheitsvotum | Mittelwert |
| Metriken | Accuracy, F1 | RMSE, MAE, R² |

### Was haben wir erwartet / Was hat überrascht?
[Eigene Reflexion der Gruppe eintragen]
